## setup

In [1]:
import numpy as np
from sklearn.metrics import roc_auc_score, precision_score, recall_score, accuracy_score
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from torch.utils.data import TensorDataset, DataLoader


/home/hia2037/EEGMotorImagery-Classification/13ENV/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import numpy as np

# Load feature arrays
X_list = []
for i in range(1, 10):
    X = np.load(fr"/home/hia2037/EEGMotorImagery-Classification/Data/X{i}.npy").astype(np.float32)
    X = X[:, None, :, :]      # Add channel dimension
    X_list.append(X)

# Combine into one dataset
X = np.concatenate(X_list, axis=0)

print("X shape:", X.shape)


# Load label arrays
y_list = []
for i in range(1, 10):
    y = np.load(fr"/home/hia2037/EEGMotorImagery-Classification/Data/X{i}L.npy")
    y_list.append(y)

# Combine labels
y = np.concatenate(y_list, axis=0)

print("y shape:", y.shape)

X shape: (5184, 1, 22, 1001)
y shape: (5184,)


In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(y)

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ShallowConvNet(nn.Module):
    def __init__(self, nb_classes, Chans=64, Samples=128, dropoutRate=0.5):
        super().__init__()

        self.conv_time = nn.Conv2d(
            in_channels=1,
            out_channels=40,
            kernel_size=(1, 13),
            bias=True
        )

        self.conv_spat = nn.Conv2d(
            in_channels=40,
            out_channels=40,
            kernel_size=(Chans, 1),
            bias=False
        )

        self.bn = nn.BatchNorm2d(
            num_features=40,
            eps=1e-5,
            momentum=0.1  # PyTorch momentum is inverse-style vs Keras
        )

        self.pool = nn.AvgPool2d(
            kernel_size=(1, 35),
            stride=(1, 7)
        )

        self.dropout = nn.Dropout(p=dropoutRate)

        # infer flattened size
        with torch.no_grad():
            x = torch.zeros(1, 1, Chans, Samples)
            x = self._features(x)
            flat_dim = x.shape[1]

        self.classifier = nn.Linear(flat_dim, nb_classes)

    def _features(self, x):
        x = self.conv_time(x)
        x = self.conv_spat(x)
        x = self.bn(x)

        # square activation
        x = x ** 2

        x = self.pool(x)

        # safe log activation
        x = torch.clamp(x, min=1e-7, max=10000)
        x = torch.log(x)

        x = self.dropout(x)
        x = torch.flatten(x, start_dim=1)
        return x

    def forward(self, x):
        """
        Expected input shape:
            (batch, 1, Chans, Samples)

        If your data is (batch, Chans, Samples, 1), convert it first.
        """
        x = self._features(x)
        x = self.classifier(x)
        return F.softmax(x, dim=1)